In [1]:
# Locate the project before importing source.
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "source" / "metrics.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from inside the ImageProcessing_Lab02 project.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch

from source.metrics import binary_metrics

y_true = torch.tensor([
    0, 1, 1, 0,
    1, 0, 1, 0
])

probabilities = torch.tensor([
    0.1, 0.9, 0.8, 0.7,
    0.2, 0.3, 0.6, 0.4
])

metrics = binary_metrics(
    y_true,
    probabilities
)

print(metrics)

{'accuracy': 0.75, 'precision': 0.75, 'recall': 0.75, 'f1': 0.75, 'tp': 3.0, 'tn': 3.0, 'fp': 1.0, 'fn': 1.0}


In [2]:
y_true = torch.tensor([0, 1, 0, 1])

probabilities = torch.tensor([
    0.1, 0.9, 0.2, 0.8
])

metrics = binary_metrics(
    y_true,
    probabilities
)

for name in [
    "accuracy",
    "precision",
    "recall",
    "f1"
]:
    assert metrics[name] == 1.0

print("Perfect classification: PASS")

Perfect classification: PASS


In [3]:
y_true = torch.tensor([0, 1, 0, 1])

probabilities = torch.tensor([
    0.9, 0.1, 0.8, 0.2
])

metrics = binary_metrics(
    y_true,
    probabilities
)

assert metrics["accuracy"] == 0.0
assert metrics["precision"] == 0.0
assert metrics["recall"] == 0.0
assert metrics["f1"] == 0.0

print("Wrong classification: PASS")

Wrong classification: PASS


In [4]:
y_true = torch.tensor([0, 0, 0, 0])

probabilities = torch.tensor([
    0.1, 0.2, 0.3, 0.4
])

metrics = binary_metrics(
    y_true,
    probabilities
)

assert metrics["accuracy"] == 1.0
assert metrics["precision"] == 0.0
assert metrics["recall"] == 0.0
assert metrics["f1"] == 0.0

print("Zero division: PASS")

Zero division: PASS


In [5]:
y_true = torch.tensor([0, 1])

probabilities = torch.tensor([
    0.49, 0.50
])

metrics = binary_metrics(
    y_true,
    probabilities,
    threshold=0.5
)

assert metrics["accuracy"] == 1.0

print("Threshold boundary: PASS")

Threshold boundary: PASS


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

torch.manual_seed(42)

y_true = torch.randint(0, 2, (1000,))
probabilities = torch.rand(1000)

predictions = (probabilities >= 0.5).long()

custom = binary_metrics(
    y_true,
    probabilities
)

reference = {
    "accuracy": accuracy_score(
        y_true, predictions
    ),
    "precision": precision_score(
        y_true, predictions, zero_division=0
    ),
    "recall": recall_score(
        y_true, predictions, zero_division=0
    ),
    "f1": f1_score(
        y_true, predictions, zero_division=0
    )
}

for name in reference:

    assert abs(
        custom[name] - reference[name]
    ) < 1e-6

    print(f"{name}: PASS")

accuracy: PASS
precision: PASS
recall: PASS
f1: PASS


In [7]:
import torch
import random

from source.config import CONFIG
from source.model import MLP, initialize_model
from source.dataset import build_loaders
from source.augmentation import build_augmentation
from source.trainer import train_epoch

torch.manual_seed(CONFIG.seed)
random.seed(CONFIG.seed)

loaders = build_loaders(
    CONFIG,
    train_transform=build_augmentation(CONFIG)
)

model = MLP(
    input_dim=CONFIG.input_dim,
    hidden_dims=CONFIG.hidden_dims,
    activation=CONFIG.activation
)

initialize_model(model, CONFIG.initialization)

model = model.to(CONFIG.device)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG.learning_rate
)

In [8]:
stats = train_epoch(
    model=model,
    loader=loaders["train"],
    optimizer=optimizer,
    criterion=criterion,
    device=CONFIG.device
)

for name, value in stats.items():
    print(f"{name}: {value:.6f}")

loss: 1.572307
accuracy: 0.550296
precision: 0.565217
recall: 0.539171
f1: 0.551887
tp: 234.000000
tn: 231.000000
fp: 180.000000
fn: 200.000000


In [9]:
import math

assert math.isfinite(stats["loss"])
assert stats["loss"] >= 0

for name in [
    "accuracy",
    "precision",
    "recall",
    "f1"
]:
    assert 0 <= stats[name] <= 1

total = (
    stats["tp"]
    + stats["tn"]
    + stats["fp"]
    + stats["fn"]
)

assert total == 845

print("Train Epoch: PASS")

Train Epoch: PASS


In [10]:
from source.trainer import evaluate

val_stats = evaluate(
    model=model,
    loader=loaders["val"],
    criterion=criterion,
    device=CONFIG.device
)

for name, value in val_stats.items():
    print(f"{name}: {value:.6f}")

loss: 0.692608
accuracy: 0.560440
precision: 0.537572
recall: 1.000000
f1: 0.699248
tp: 93.000000
tn: 9.000000
fp: 80.000000
fn: 0.000000


In [11]:
import math

assert math.isfinite(val_stats["loss"])
assert val_stats["loss"] >= 0

for name in [
    "accuracy",
    "precision",
    "recall",
    "f1"
]:
    assert 0 <= val_stats[name] <= 1

total = (
    val_stats["tp"]
    + val_stats["tn"]
    + val_stats["fp"]
    + val_stats["fn"]
)

assert total == 182

print("Validation Epoch: PASS")

Validation Epoch: PASS


In [12]:
before = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
}

evaluate(
    model,
    loaders["val"],
    criterion,
    CONFIG.device
)

for name, param in model.named_parameters():

    assert torch.equal(
        before[name],
        param.detach()
    )

print("Parameter Unchanged: PASS")

Parameter Unchanged: PASS


In [13]:
stats1 = evaluate(
    model,
    loaders["val"],
    criterion,
    CONFIG.device
)

stats2 = evaluate(
    model,
    loaders["val"],
    criterion,
    CONFIG.device
)

for name in stats1:

    assert math.isclose(
        stats1[name],
        stats2[name],
        rel_tol=1e-6,
        abs_tol=1e-6
    )

print("Deterministic Evaluation: PASS")

Deterministic Evaluation: PASS


In [14]:
model.train()

evaluate(
    model,
    loaders["val"],
    criterion,
    CONFIG.device
)

assert not model.training

print("Evaluation Mode: PASS")

Evaluation Mode: PASS


In [15]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

for epoch in range(5):

    train_stats = train_epoch(
        model,
        loaders["train"],
        optimizer,
        criterion,
        CONFIG.device
    )

    val_stats = evaluate(
        model,
        loaders["val"],
        criterion,
        CONFIG.device
    )

    history["train_loss"].append(
        train_stats["loss"]
    )

    history["val_loss"].append(
        val_stats["loss"]
    )

    history["train_accuracy"].append(
        train_stats["accuracy"]
    )

    history["val_accuracy"].append(
        val_stats["accuracy"]
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_stats['loss']:.4f} | "
        f"Val Loss: {val_stats['loss']:.4f} | "
        f"Train Acc: {train_stats['accuracy']:.2%} | "
        f"Val Acc: {val_stats['accuracy']:.2%}"
    )

Epoch 01 | Train Loss: 0.6654 | Val Loss: 0.6122 | Train Acc: 61.66% | Val Acc: 56.04%
Epoch 02 | Train Loss: 0.5368 | Val Loss: 0.6553 | Train Acc: 72.07% | Val Acc: 67.03%
Epoch 03 | Train Loss: 0.5451 | Val Loss: 0.5860 | Train Acc: 71.01% | Val Acc: 73.63%
Epoch 04 | Train Loss: 0.4626 | Val Loss: 0.4822 | Train Acc: 77.99% | Val Acc: 77.47%
Epoch 05 | Train Loss: 0.4817 | Val Loss: 0.5445 | Train Acc: 76.69% | Val Acc: 66.48%


In [16]:
import random
import torch

from source.trainer import fit

torch.manual_seed(CONFIG.seed)
random.seed(CONFIG.seed)

loaders = build_loaders(
    CONFIG,
    train_transform=build_augmentation(CONFIG)
)

model = MLP(
    input_dim=CONFIG.input_dim,
    hidden_dims=CONFIG.hidden_dims,
    activation=CONFIG.activation
)

initialize_model(model, CONFIG.initialization)

model = model.to(CONFIG.device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG.learning_rate
)

criterion = torch.nn.BCEWithLogitsLoss()

In [17]:
checkpoint_path = (
    CONFIG.results_dir
    / "checkpoints"
    / "smoke_best_model.pt"
)

history_path = (
    CONFIG.results_dir
    / "histories"
    / "smoke_training_history.json"
)

history = fit(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    optimizer=optimizer,
    criterion=criterion,
    device=CONFIG.device,
    epochs=5,
    checkpoint_path=checkpoint_path,
    history_path=history_path
)

Epoch 1/5 | train loss: 1.5723 | val loss: 0.6926 | val F1: 0.6992
Epoch 2/5 | train loss: 0.6654 | val loss: 0.6122 | val F1: 0.4118
Epoch 3/5 | train loss: 0.5368 | val loss: 0.6553 | val F1: 0.7541
Epoch 4/5 | train loss: 0.5451 | val loss: 0.5860 | val F1: 0.7876
Epoch 5/5 | train loss: 0.4626 | val loss: 0.4822 | val F1: 0.8000


In [18]:
assert checkpoint_path.is_file()
assert history_path.is_file()

assert len(history["train"]) == 5
assert len(history["val"]) == 5

expected_best_epoch = min(
    history["val"],
    key=lambda stats: stats["loss"]
)["epoch"]

assert history["best_epoch"] == expected_best_epoch

checkpoint = torch.load(
    checkpoint_path,
    map_location=CONFIG.device,
    weights_only=True
)

assert checkpoint["epoch"] == expected_best_epoch

print("Best epoch:", expected_best_epoch)
print("Best validation loss:", history["best_val_loss"])

print("Checkpoint Management: PASS")

Best epoch: 5
Best validation loss: 0.48224209003396085
Checkpoint Management: PASS


In [19]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

restored_stats = evaluate(
    model,
    loaders["val"],
    criterion,
    CONFIG.device
)

assert abs(
    restored_stats["loss"]
    - history["best_val_loss"]
) < 1e-6

print("Restored validation loss:", restored_stats["loss"])

print("Best Model Restoration: PASS")

Restored validation loss: 0.48224209003396085
Best Model Restoration: PASS


In [20]:
import importlib

# Reload inference code after edits to source/predict.py.
importlib.reload(importlib.import_module("source.predict"))

from source.predict import (
    load_run_config,
    load_model,
    predict,
    evaluate_checkpoint
)

run_name = "smoke_cli"

config = load_run_config(run_name)

checkpoint_path = (
    config.results_dir
    / "checkpoints"
    / f"{run_name}.pt"
)

print(config)

Config(root=PosixPath('/home/kjwyy/Documents/code/IMGproccess/BT-TH/ImageProcessing_Lab02'), seed=42, image_size=(64, 64), image_channels=3, num_classes=2, hidden_dims=[128, 64], batch_size=32, learning_rate=0.001, epochs=3, activation='relu', initialization='he_normal', optimizer='Adam', loss='BCEWithLogitsLoss', device='cpu', num_workers=0)


In [21]:
import pandas as pd

df = pd.read_csv(
    config.splits_dir / "val.csv"
)

row = df.iloc[0]

image_path = config.root / row["image_path"]

result = predict(
    image_path,
    checkpoint_path,
    config
)

print("Ground truth:", row["class_name"])
print("Prediction:", result)

Ground truth: horses
Prediction: {'label': 0, 'class_name': 'horse', 'human_probability': 0.14640845358371735}


In [22]:
assert result["label"] in (0, 1)

assert 0 <= result["human_probability"] <= 1

assert result["label"] == int(
    result["human_probability"] >= 0.5
)

print("Single Image Prediction: PASS")

Single Image Prediction: PASS


In [23]:
stats = evaluate_checkpoint(
    checkpoint_path,
    config,
    split="val"
)

for name, value in stats.items():
    print(f"{name}: {value:.6f}")

loss: 0.612168
accuracy: 0.560440
precision: 0.651163
recall: 0.301075
f1: 0.411765
tp: 28.000000
tn: 74.000000
fp: 15.000000
fn: 65.000000


In [24]:
import json

history_path = (
    config.results_dir
    / "histories"
    / f"{run_name}.json"
)

with history_path.open() as file:
    history = json.load(file)

assert abs(
    stats["loss"] - history["best_val_loss"]
) < 1e-6

assert (
    stats["tp"]
    + stats["tn"]
    + stats["fp"]
    + stats["fn"]
) == 182

print("Checkpoint Evaluation: PASS")

Checkpoint Evaluation: PASS


In [25]:
from source.visualization import (
    plot_history,
    plot_confusion_matrix
)

plot_history(
    history,
    CONFIG.results_dir
    / "figures/smoke_training_curves.png"
)

plot_confusion_matrix(
    stats,
    CONFIG.results_dir
    / "figures/smoke_confusion_matrix.png"
)

print("Visualization completed")

Visualization completed
